In [1]:
print("HELLO")

HELLO


In [4]:
import os
from pathlib import Path

cwd = Path(os.getcwd())
DOCS = Path(cwd.parent, "docs")

DOCS

WindowsPath('c:/Users/tejas/Desktop/Code/Hackathon/Buildonomics/AI/docs')

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader


file1 = Path(DOCS, "GS.pdf")
loader = PyMuPDFLoader(file_path=file1)

c:\Users\tejas\Desktop\Code\Hackathon\Buildonomics\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
RCT = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

text = loader.load()
texts = RCT.split_text(text[0].page_content)

In [35]:
type(text[0])

langchain_core.documents.base.Document

In [ ]:
from langchain_core.documents import Document

resume = [Document(page_content=text) for text in texts]
resume = text[0]


[Document(metadata={}, page_content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004'),
 Document(metadata={}, page_content='|\nLeetCode: leetcode.com/u/Tejasisnothere\nProfessional Summary\nB.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation'),
 Document(metadata={}, page_content='in Data Structures, Algorithms, and quantitative problem-solving (400+ LeetCode problems solved, JEE Percentile:'),
 Document(metadata={}, page_content='96.8). Experienced in building analytical and AI-driven systems for financial and forecasting use cases, including a'),
 Document(metadata={}, page_content='compliance-analysis engine and a time-series demand-forecasting platform, alongside multi-agent AI pipelines using'),
 Document(metadata={}, page_content='LangChain and LangGraph. Comfor

In [41]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes = [
    "Person", "Institution", "Degree", "Skill",
    "Project", "Certification", "Organization", "Achievement",
    ],
allowed_relationships= [
    ("Person", "STUDIED_AT", "Institution"),
    ("Person", "HOLDS_DEGREE", "Degree"),
    ("Person", "HAS_SKILL", "Skill"),
    ("Person", "BUILT", "Project"),
    ("Project", "USES_TECHNOLOGY", "Skill"),
    ("Person", "EARNED_CERTIFICATION", "Certification"),
    ("Certification", "ISSUED_BY", "Organization"),
    ("Person", "ACHIEVED", "Achievement"),
]
 ,
)
graph_documents = llm_transformer.convert_to_graph_documents(resume)

In [42]:
from langchain_neo4j import Neo4jGraph


graph_store = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="Tjtk2004!", database="test")
graph_store.add_graph_documents(graph_documents=graph_documents)

In [43]:
graph_documents

[GraphDocument(nodes=[Node(id='Tejas Kadam', type='Person', properties={})], relationships=[], source=Document(metadata={}, page_content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004')),
 GraphDocument(nodes=[Node(id='Tejasisnothere', type='Person', properties={}), Node(id='Vellore Institute Of Technology', type='Institution', properties={}), Node(id='B.Tech Computer Science And Business Systems', type='Degree', properties={})], relationships=[Relationship(source=Node(id='Tejasisnothere', type='Person', properties={}), target=Node(id='Vellore Institute Of Technology', type='Institution', properties={}), type='STUDIED_AT', properties={}), Relationship(source=Node(id='Tejasisnothere', type='Person', properties={}), target=Node(id='B.Tech Computer Science And Business Systems', type='Degree', properties={}), type='HOLDS_DEGREE',

In [50]:
"""
Hardened resume -> knowledge graph pipeline.

Fixes applied vs. the earlier version:
  1. Explicit (source, relation, target) tuples in allowed_relationships,
     so the LLM can't emit orphan edges that skip Person.
  2. A canonical "anchor_id" injected into the prompt so every extraction
     pass ties back to the SAME Person node (prevents duplicate Person nodes
     across resume vs. README extraction runs).
  3. Name normalization before writing, so "Tejas Kadam" / "TEJAS KADAM" /
     " tejas kadam " never create separate nodes.
  4. add_graph_documents uses MERGE semantics under the hood via Neo4jGraph,
     but only if node ids match exactly post-normalization -- so normalization
     has to happen BEFORE convert_to_graph_documents, not after.
  5. A post-write connectivity check that flags any node not reachable from
     Person, so silent orphans get caught immediately instead of discovered
     later by eyeballing the graph.
"""

import os
import re
import json
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph

load_dotenv()


# ---------------------------------------------------------------------------
# 1. Normalization -- do this BEFORE any extraction, not after
# ---------------------------------------------------------------------------
def normalize_text(text: str) -> str:
    """Collapse whitespace only -- do NOT lowercase the whole resume,
    since that would mangle proper nouns like project/company names
    that the LLM uses verbatim as node ids."""
    return re.sub(r"\s+", " ", text).strip()


def canonical_person_name(raw_name: str) -> str:
    """Normalize just the person's name to a single consistent form,
    used to force every extraction pass to name the Person node identically."""
    return " ".join(word.capitalize() for word in raw_name.strip().split())


# ---------------------------------------------------------------------------
# 2. Schema: explicit (source, relation, target) triples
# ---------------------------------------------------------------------------
ALLOWED_NODES = [
    "Person", "Institution", "Degree", "Skill",
    "Project", "Certification", "Organization", "Achievement",
]

ALLOWED_RELATIONSHIPS = [
    ("Person", "STUDIED_AT", "Institution"),
    ("Person", "HOLDS_DEGREE", "Degree"),
    ("Person", "HAS_SKILL", "Skill"),
    ("Person", "BUILT", "Project"),
    ("Project", "USES_TECHNOLOGY", "Skill"),
    ("Person", "EARNED_CERTIFICATION", "Certification"),
    ("Certification", "ISSUED_BY", "Organization"),
    ("Person", "ACHIEVED", "Achievement"),
]


def build_transformer(llm, anchor_name: str) -> LLMGraphTransformer:
    """anchor_name gets baked into the additional_instructions so every
    extraction pass -- resume AND later README passes -- refers to the
    same Person node id, instead of letting the LLM re-derive the name
    from context each time (which is what caused the duplicate Person)."""
    return LLMGraphTransformer(
        llm=llm,
        allowed_nodes=ALLOWED_NODES,
        allowed_relationships=ALLOWED_RELATIONSHIPS,
        # node_properties=["description"],  # lets Project nodes carry a description property
        additional_instructions=(
            f"The central Person in this document is always named exactly "
            f"'{anchor_name}'. Use this exact string as the Person node's id "
            f"in every relationship. Every other node must connect back to "
            f"this Person, directly or indirectly -- do not emit a relationship "
            f"between two non-Person nodes without also connecting at least one "
            f"of them to Person elsewhere in your output."
        ),
    )


# ---------------------------------------------------------------------------
# 3. Extraction
# ---------------------------------------------------------------------------
def extract_resume_graph(resume_text: str, person_name: str, llm) -> list:
    anchor = canonical_person_name(person_name)
    text = normalize_text(resume_text)
    doc = Document(page_content=text, metadata={"source": "resume", "person": anchor})

    transformer = build_transformer(llm, anchor)
    return transformer.convert_to_graph_documents([doc])


# ---------------------------------------------------------------------------
# 4. Write -- MERGE-safe because Neo4jGraph.add_graph_documents already
#    uses MERGE on node id under the hood, so as long as ids are normalized
#    consistently (step 1-3), re-running this is idempotent.
# ---------------------------------------------------------------------------
def write_graph(graph: Neo4jGraph, graph_documents: list):
    graph.add_graph_documents(
        graph_documents,
        include_source=True,
        baseEntityLabel=True,
    )


# ---------------------------------------------------------------------------
# 5. Post-write validation -- catches orphans and duplicate Person nodes
#    immediately instead of relying on eyeballing the Explore view.
# ---------------------------------------------------------------------------
def validate_graph(graph: Neo4jGraph, person_name: str):
    anchor = canonical_person_name(person_name)

    # a) duplicate Person check
    dupes = graph.query(
        "MATCH (p:Person) RETURN p.id AS id, count(*) AS c"
    )
    person_nodes = [r for r in dupes if r["id"]]
    if len(person_nodes) > 1:
        print(f"WARNING: found {len(person_nodes)} Person nodes, expected 1:")
        for r in person_nodes:
            print(f"  - {r['id']}")

    # b) orphan check -- anything not reachable from Person
    orphans = graph.query(
        """
        MATCH (p:Person {id: $name})
        CALL (p) {
          MATCH (p)-[*]-(reachable)
          RETURN collect(DISTINCT reachable) AS reached
        }
        MATCH (n)
        WHERE NOT n IN reached AND n <> p AND n:Person = false
        RETURN labels(n) AS labels, n.id AS id
        """,
        params={"name": anchor},
    )
    if orphans:
        print(f"WARNING: {len(orphans)} node(s) not connected to Person:")
        for r in orphans:
            print(f"  - {r['labels']}: {r['id']}")
    else:
        print("Graph connectivity OK -- all nodes reachable from Person.")


# ---------------------------------------------------------------------------
# 6. Full run
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    llm = ChatGroq(model="openai/gpt-oss-120b")

    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password="Tjtk2004!",
        database=os.environ.get("NEO4J_DATABASE", "test"),
        refresh_schema=False,
    )

    resume_text = text[0].page_content   # your PDF-extracted text[0].page_content goes here
    person_name = "Tejas Kadam"  # exact name as it should appear as the anchor

    graph_documents = extract_resume_graph(resume_text, person_name, llm)
    write_graph(graph, graph_documents)
    validate_graph(graph, person_name)

BadRequestError: Error code: 400 - {'error': {'message': 'Failed to parse tool call arguments as JSON', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "DynamicGraph", "arguments": {\n  "nodes": [\n    {"id": "Tejas Kadam", "type": "Person"},\n    {"id": "Vellore Institute of Technology", "type": "Institution"},\n    {"id": "Ramsheth Thakur Public School", "type": "Institution"},\n    {"id": "Bachelor of Technology in Computer Science and Business Systems", "type": "Degree"},\n    {"id": "Python", "type": "Skill"},\n    {"id": "C++", "type": "Skill"},\n    {"id": "SQL", "type": "Skill"},\n    {"id": "JavaScript", "type": "Skill"},\n    {"id": "Data Structures & Algorithms", "type": "Skill"},\n    {"id": "OOP", "type": "Skill"},\n    {"id": "System Design", "type": "Skill"},\n    {"id": "Full-Stack Development", "type": "Skill"},\n    {"id": "Time-Series Forecasting", "type": "Skill"},\n    {"id": "NLP", "type": "Skill"},\n    {"id": "Reinforce

In [ ]:
from pydantic import BaseModel

class UserInfo()